# LLM Counterfactual Fairness Case Study

This notebook demonstrates fairpipe's counterfactual fairness probe using **committed
live-recorded responses** from Anthropic Claude Haiku — replayed from cache with zero API calls.

We measure **counterfactual fairness divergence**: how much model outputs change when a
demographic attribute in the prompt is swapped (here, gender-coded hiring recommendations).

## Important limitations (read before interpreting numbers)

- **n=1 prompt per group** in this fixture (`woman`, `man`, `nonbinary`). That is a
  **code-correctness smoke test**, not a sample large enough to draw conclusions about
  Claude Haiku's behavior in production.
- The default LLM eval threshold is **`min_group_size=5`** (below the classifier's 30
  because each sample is a paid API call). At default settings this fixture is **below
  threshold** and the metric correctly returns **`nan`** — same silent exclude semantics
  as `FairnessAnalyzer` / `NativeAdapter`.
- The illustrative divergence shown below uses **`allow_small_samples=True`**. That override
  is explicitly labeled as non-production; do **not** treat **0.187** as a finding about
  the model.
- **No confidence interval** is reported here: with one prompt per group, bootstrap CI
  would resample only three pairwise scalars and overstate precision.

In [ ]:
import math

from fairness_pipeline_dev_toolkit.llm_evals import (
    DEFAULT_LLM_MIN_GROUP_SIZE,
    default_recorded_counterfactual_config,
    run_llm_eval,
)

config = default_recorded_counterfactual_config()

# Production-default path: guard excludes n=1 groups → nan (classifier-parity semantics).
blocked = run_llm_eval(config, with_ci=False)
blocked_metric = blocked.metrics["counterfactual_fairness_divergence"]
print(f"Default min_group_size={DEFAULT_LLM_MIN_GROUP_SIZE}")
print(f"Metric at default threshold: {blocked_metric.value}")
print(f"Eligible n_per_group: {blocked_metric.n_per_group}")
assert math.isnan(blocked_metric.value)
print("Guard correctly blocked below-threshold fixture (nan, empty eligible groups).")

In [ ]:
# Illustrative-only override: below production threshold — NOT a model behavior claim.
result = run_llm_eval(config, allow_small_samples=True, with_ci=False)
metric = result.metrics["counterfactual_fairness_divergence"]
COUNTERFACTUAL_DIVERGENCE = metric.value
print("ILLUSTRATIVE RUN (allow_small_samples=True — not production-grade)")
print(f"Counterfactual fairness divergence: {COUNTERFACTUAL_DIVERGENCE:.4f}")
print(f"n_per_group: {metric.n_per_group}")
print(f"Provider: {config.provider} / {config.model} (cache replay)")

In [ ]:
assert COUNTERFACTUAL_DIVERGENCE > 0.0, "Smoke test: non-zero divergence under illustrative override"
print("Notebook smoke-test check passed (not a production fairness finding).")